In [1]:
suppressMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(pheatmap)
  library(cowplot)
  library(RColorBrewer)
  library(dplyr)
  library(ComplexHeatmap)
  library(RColorBrewer)
  cols = c(brewer.pal(9, "Set1"),brewer.pal(8,"Set2")[1:8],brewer.pal(12,"Paired")[1:12],brewer.pal(8,"Dark2")[1:8],brewer.pal(8,"Accent"),brewer.pal(12, "Set3"),brewer.pal(9,"Pastel1"),brewer.pal(8,"Pastel2"))
  cols = c(cols,cols)
})

Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”
Warning message:
“package ‘dplyr’ was built under R version 4.2.3”


In [2]:
library(future)
options(future.globals.maxSize = 50 * 1024^3)
plan(multisession, workers=16)

In [3]:
### GroupA  con
merge.rds1 = readRDS(paste0("/data/Bin200.merge.debatch.rds"))

In [5]:
#### GroupⅢ  case
merge.rds2 = readRDS(paste0("/data/Bin200.GroupⅢ.merge.debatch.rds"))

In [ ]:
#### Layer Boundary Entropy Value

In [9]:
head(merge.rds1)[1:3,]
merge.rds1$Bayes20 = as.character(merge.rds1$Bayes20)
merge.rds1$Bayes16 = as.character(merge.rds1$Bayes16)
head(merge.rds2)[1:3,]

,orig.ident,nCount_Spatial,nFeature_Spatial,area,percent.mt,nCount_SCT,nFeature_SCT,SCT_snn_res.0.5,SCT_snn_res.0.8,SCT_snn_res.1.2,⋯,Bayes14,Bayes15,Bayes17,Bayes18,Bayes19,Bayes21,Bayes22,Bayes23,Bayes.anno16,sublayer
,<chr>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<int>,<fct>,<chr>,<chr>,⋯,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<chr>
T756_BIN200_27,BIN200,591,309,L6,18.78173,23535,7266,7,4,8,⋯,12,14,14,15,16,17,21,22,L1,L1_a
T756_BIN200_28,BIN200,1827,809,L6,24.52107,24052,6444,5,4,8,⋯,1,1,1,1,4,1,1,1,WM,WM_a
T756_BIN200_29,BIN200,3296,1074,WM,39.13835,23920,5625,5,4,8,⋯,1,1,1,1,4,1,1,1,WM,WM_a


,orig.ident,nCount_Spatial,nFeature_Spatial,area,nCount_SCT,nFeature_SCT,SCT_snn_res.0.5,seurat_clusters,coor_y,coor_x,T_names,cellid,row,col,Bayes16,Bayes20,Bayes24,Bayes.anno20
,<chr>,<dbl>,<int>,<chr>,<dbl>,<int>,<fct>,<fct>,<dbl>,<dbl>,<chr>,<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>
T716_BIN2_1353,BIN2,2735,1242,L1,10662,2704,5,5,9,93,T716,T716_BIN2_1353,9,93,11,13,22,L1
T716_BIN2_1354,BIN2,8753,3153,L1,11560,3168,5,5,9,94,T716,T716_BIN2_1354,9,94,9,11,12,L1
T716_BIN2_1355,BIN2,3734,1663,L1,10840,2569,5,5,9,95,T716,T716_BIN2_1355,9,95,9,11,12,L1


In [10]:
Idents(merge.rds1)="area"
merge.rds1 = subset(merge.rds1,idents=c("L1","L2","L3","L4","L5","L6"))

In [11]:
table(merge.rds1$T_names)


T756 T928 T989 T993 T996 
2101 1394 2922 4689 5041 

In [ ]:
options(repr.plot.width=15, repr.plot.height=5) 
p = DimPlot(merge.rds1,reduction="stereo",group.by = 'Bayes20',label = TRUE,label.size = 0.6,pt.size = 0.001,raster=FALSE,cols = cols)+coord_fixed()+ theme_void() +
          theme(axis.text = element_blank(),
                axis.ticks = element_blank(),
                axis.title = element_blank(),
                panel.background = element_rect(fill = "black"))
p

In [13]:
Idents(merge.rds2)="area"
merge.rds2 = subset(merge.rds2,idents=c("L1","L2","L3","L4","L5","L6"))

In [ ]:
options(repr.plot.width=15, repr.plot.height=5) 
p = DimPlot(merge.rds2,reduction="stereo",group.by = 'Bayes20',label = TRUE,label.size = 0.6,pt.size = 0.001,raster=FALSE,cols = cols)+coord_fixed()+ theme_void() +
          theme(axis.text = element_blank(),
                axis.ticks = element_blank(),
                axis.title = element_blank(),
                panel.background = element_rect(fill = "black"))
p

In [17]:
# Shannon entropy
calc_entropy <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_real_)
  p <- table(as.character(x)) / length(x)
  p <- p[p > 0]
  -sum(p * log2(p))
}

# Normalized Entropy: 0–1, making comparisons more convenient.
calc_norm_entropy <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_real_)
  p <- table(as.character(x)) / length(x)
  p <- p[p > 0]
  if (length(p) <= 1) return(0)
  -sum(p * log2(p)) / log2(length(p))
}

# Constructing Adjacency Edges (Based on Rows/Columns)
make_spatial_edges <- function(df, row_col = c("row", "col"), use_diagonal = TRUE) {
  rcol <- row_col[1]
  ccol <- row_col[2]
  
  coords <- df[, c(rcol, ccol)]
  coords$cellid <- rownames(df)
  
  key <- paste(coords[[rcol]], coords[[ccol]], sep = "_")
  names(key) <- coords$cellid
  key2cell <- setNames(coords$cellid, key)
  
  if (use_diagonal) {
    offsets <- expand.grid(dr = -1:1, dc = -1:1) %>%
      filter(!(dr == 0 & dc == 0))
  } else {
    offsets <- data.frame(
      dr = c(-1, 1, 0, 0),
      dc = c(0, 0, -1, 1)
    )
  }
  
  edge_list <- list()
  idx <- 1
  
  for (i in seq_len(nrow(coords))) {
    r <- coords[[rcol]][i]
    c <- coords[[ccol]][i]
    from <- coords$cellid[i]
    
    for (j in seq_len(nrow(offsets))) {
      nr <- r + offsets$dr[j]
      nc <- c + offsets$dc[j]
      nkey <- paste(nr, nc, sep = "_")
      
      if (nkey %in% names(key2cell)) {
        to <- key2cell[[nkey]]
        edge_list[[idx]] <- c(from, to)
        idx <- idx + 1
      }
    }
  }
  
  if (length(edge_list) == 0) {
    return(data.frame(from = character(), to = character()))
  }
  
  edges <- do.call(rbind, edge_list) %>%
    as.data.frame(stringsAsFactors = FALSE)
  colnames(edges) <- c("from", "to")
  
  edges2 <- t(apply(edges, 1, sort))
  edges2 <- unique(edges2)
  edges2 <- as.data.frame(edges2, stringsAsFactors = FALSE)
  colnames(edges2) <- c("from", "to")
  
  edges2
}

In [18]:
compute_area_based_disorder_one_sample <- function(
    subdf,
    area_col = "area",
    cluster_col = "Bayes20",
    row_col = c("row", "col"),
    use_diagonal = TRUE
) {
  subdf <- subdf %>%
    filter(!is.na(.data[[area_col]]), !is.na(.data[[cluster_col]])) %>%
    mutate(
      area_use = as.character(.data[[area_col]]),
      cluster_use = as.character(.data[[cluster_col]])
    )
  
  if (nrow(subdf) == 0) return(NULL)
  
  # ---------- Metric 1/2: Bayes20 Entropy / Purity within each area ----------
  area_metrics <- subdf %>%
    group_by(area_use) %>%
    summarise(
      n_spots = n(),
      bayes_entropy = calc_entropy(cluster_use),
      bayes_norm_entropy = calc_norm_entropy(cluster_use),
      bayes_purity = max(table(cluster_use)) / n(),
      n_bayes_clusters = n_distinct(cluster_use),
      .groups = "drop"
    )
  
  # Sample-Level Weighted Aggregation
  weighted_entropy <- weighted.mean(area_metrics$bayes_entropy, w = area_metrics$n_spots, na.rm = TRUE)
  weighted_norm_entropy <- weighted.mean(area_metrics$bayes_norm_entropy, w = area_metrics$n_spots, na.rm = TRUE)
  weighted_purity <- weighted.mean(area_metrics$bayes_purity, w = area_metrics$n_spots, na.rm = TRUE)
  
  # ---------- Metric 3: Global Consistency (area vs. Bayes20) ----------
  ari <- tryCatch(
    mclust::adjustedRandIndex(subdf$area_use, subdf$cluster_use),
    error = function(e) NA_real_
  )
  
  # ---------- Metric 4: Local Neighborhood Bayes20 Inconsistency Rate within the Same Area----------
  edges <- make_spatial_edges(subdf, row_col = row_col, use_diagonal = use_diagonal)
  
  if (nrow(edges) == 0) {
    local_discordance <- NA_real_
  } else {
    edge_df <- edges %>%
      mutate(
        area_from = subdf[from, "area_use"],
        area_to = subdf[to, "area_use"],
        bayes_from = subdf[from, "cluster_use"],
        bayes_to = subdf[to, "cluster_use"]
      ) %>%
      filter(area_from == area_to)
    
    if (nrow(edge_df) == 0) {
      local_discordance <- NA_real_
    } else {
      local_discordance <- mean(edge_df$bayes_from != edge_df$bayes_to, na.rm = TRUE)
    }
  }
  
  list(
    area_level = area_metrics,
    sample_level = data.frame(
      weighted_entropy = weighted_entropy,
      weighted_norm_entropy = weighted_norm_entropy,
      weighted_purity = weighted_purity,
      ARI_area_Bayes20 = ari,
      within_area_local_discordance = local_discordance
    )
  )
}

In [19]:
compute_area_based_disorder_object <- function(
    seu,
    area_col = "area",
    cluster_col = "Bayes20",
    sample_col = "T_names",
    row_col = c("row", "col"),
    use_diagonal = TRUE,
    object_name = "group"
) {
  meta <- seu@meta.data
  meta$cellid <- rownames(meta)
  
  need_cols <- c(area_col, cluster_col, sample_col, row_col)
  miss_cols <- setdiff(need_cols, colnames(meta))
  if (length(miss_cols) > 0) {
    stop("Missing columns: ", paste(miss_cols, collapse = ", "))
  }
  
  rownames(meta) <- rownames(seu@meta.data)
  
  res_list <- lapply(split(meta, meta[[sample_col]]), function(subdf) {
    ans <- compute_area_based_disorder_one_sample(
      subdf = subdf,
      area_col = area_col,
      cluster_col = cluster_col,
      row_col = row_col,
      use_diagonal = use_diagonal
    )
    if (is.null(ans)) return(NULL)
    
    area_df <- ans$area_level
    area_df[[sample_col]] <- unique(subdf[[sample_col]])[1]
    
    sample_df <- ans$sample_level
    sample_df[[sample_col]] <- unique(subdf[[sample_col]])[1]
    
    list(area_level = area_df, sample_level = sample_df)
  })
  
  area_level_all <- bind_rows(lapply(res_list, `[[`, "area_level")) %>%
    mutate(object_group = object_name)
  
  sample_level_all <- bind_rows(lapply(res_list, `[[`, "sample_level")) %>%
    rename(T_names = .data[[sample_col]]) %>%
    mutate(object_group = object_name)
  
  list(
    area_level = area_level_all,
    sample_level = sample_level_all
  )
}

In [20]:
res_area1 <- compute_area_based_disorder_object(
  seu = merge.rds1,
  area_col = "area",
  cluster_col = "Bayes20",
  sample_col = "T_names",
  row_col = c("row", "col"),
  use_diagonal = TRUE,
  object_name = "merge.rds1"
)

res_area2 <- compute_area_based_disorder_object(
  seu = merge.rds2,
  area_col = "area",
  cluster_col = "Bayes20",
  sample_col = "T_names",
  row_col = c("row", "col"),
  use_diagonal = TRUE,
  object_name = "merge.rds2"
)

Warning message:
“Use of .data in tidyselect expressions was deprecated in tidyselect 1.2.0.
ℹ Please use `all_of(var)` (or `any_of(var)`) instead of `.data[[var]]`”


In [21]:
sample_disorder_all <- bind_rows(res_area1$sample_level, res_area2$sample_level)
sample_disorder_all

weighted_entropy,weighted_norm_entropy,weighted_purity,ARI_area_Bayes20,within_area_local_discordance,T_names,object_group
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1.682780,0.6117315,0.5202285,0.4053484,0.3022825,T756,merge.rds1
1.780849,0.6288904,0.4964132,0.3999122,0.4230385,T928,merge.rds1
1.816402,0.6615068,0.4192334,0.3519291,0.3399631,T989,merge.rds1
1.943563,0.7034694,0.4252506,0.3402599,0.3480386,T993,merge.rds1
1.673312,0.6145203,0.5125967,0.4144850,0.2950962,T996,merge.rds1
2.607786,0.7495687,0.3416724,0.1835483,0.3215852,T716,merge.rds2
2.152792,0.6456768,0.4478035,0.2408557,0.3574680,T734,merge.rds2
2.357503,0.7038950,0.3903779,0.1965884,0.3599090,T760,merge.rds2
2.642153,0.7648118,0.3135593,0.1572965,0.3961921,T806,merge.rds2


In [22]:
wilcox.test(weighted_norm_entropy ~ object_group, data = sample_disorder_all)


	Wilcoxon rank sum exact test

data:  weighted_norm_entropy by object_group
W = 2, p-value = 0.03175
alternative hypothesis: true location shift is not equal to 0


In [23]:
wilcox.test(weighted_purity ~ object_group, data = sample_disorder_all)


	Wilcoxon rank sum exact test

data:  weighted_purity by object_group
W = 23, p-value = 0.03175
alternative hypothesis: true location shift is not equal to 0


In [24]:
wilcox.test(within_area_local_discordance ~ object_group, data = sample_disorder_all)


	Wilcoxon rank sum exact test

data:  within_area_local_discordance by object_group
W = 9, p-value = 0.5476
alternative hypothesis: true location shift is not equal to 0


In [ ]:
#### Bayes20 Entropy within the Area
p1 <- wilcox.test(weighted_norm_entropy ~ object_group, data = sample_disorder_all)$p.value

plot1 = ggplot(sample_disorder_all, aes(x = object_group, y = weighted_norm_entropy, fill = object_group)) +
  geom_boxplot(width = 0.5, outlier.shape = NA) +
  geom_jitter(width = 0.08, size = 2) +
  geom_text(aes(label = T_names), hjust = -0.1, size = 3) +
  theme_bw() +
  labs(
    x = "",
    y = "Weighted normalized Bayes20 entropy within area",
    title = paste0("Area-based Bayes20 disorder\nWilcoxon p = ", signif(p1, 3))
  )
plot1

In [29]:
ggsave(plot=plot1,"Bayes20.entropy.area.pdf")

Saving 6.67 x 6.67 in image


In [32]:
p1_adjusted

[1] 0.03174603

In [ ]:
# Wilcoxon test
p1 <- wilcox.test(weighted_norm_entropy ~ object_group, data = sample_disorder_all)$p.value

# If multiple comparisons are involved, perform FDR correction.
p1_adjusted <- p.adjust(p1, method = "BH")

# plot
plot1 = ggplot(sample_disorder_all, aes(x = object_group, y = weighted_norm_entropy, fill = object_group)) +
  geom_boxplot(width = 0.5, outlier.shape = NA) +
  geom_jitter(width = 0.08, size = 2) +
  geom_text(aes(label = T_names), hjust = -0.1, size = 3) +
  theme_bw() +
  labs(
    x = "",
    y = "Weighted normalized Bayes20 entropy within area",
    title = paste0("Area-based Bayes20 disorder\nWilcoxon p = ", signif(p1_adjusted, 3), 
                   " (Benjamini-Hochberg FDR adjusted)")
  )
print(plot1)

In [ ]:
### Local Inconsistency Rate within the Same Area
p3 <- wilcox.test(within_area_local_discordance ~ object_group, data = sample_disorder_all)$p.value
p3_adjusted <- p.adjust(p3, method = "BH")
p3_adjusted
plot3 = ggplot(sample_disorder_all, aes(x = object_group, y = within_area_local_discordance, fill = object_group)) +
  geom_boxplot(width = 0.5, outlier.shape = NA) +
  geom_jitter(width = 0.08, size = 2) +
  geom_text(aes(label = T_names), hjust = -0.1, size = 3) +
  theme_bw() +
  labs(
    x = "",
    y = "Within-area local Bayes20 discordance",
    title = paste0("Local Bayes20 discordance within area\nWilcoxon p = ", signif(p3, 3))
  )
plot3

In [ ]:
####### Comparison of Each Layer: If you want to see specifically which areas are more cluttered
area_disorder_all <- bind_rows(res_area1$area_level, res_area2$area_level)

plot4 = ggplot(area_disorder_all, aes(x = area_use, y = bayes_norm_entropy, fill = object_group)) +
  geom_boxplot(outlier.shape = NA, position = position_dodge(width = 0.75)) +
  geom_jitter(position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75), size = 1) +
  theme_bw() +
  labs(
    x = "Morphology-defined area",
    y = "Bayes20 normalized entropy",
    title = "Area-specific Bayes20 disorder"
  )
plot4

In [35]:
ggsave(plot=plot4,"diff.area.entropy.pdf",width=10)

Saving 10 x 6.67 in image
